# US States Energy Data

This dataset contains all energy data of the US and its states and territories.
The dataset is sourced from EIA's State Energy Data System.

## Dataset

Update: June 10th, 2022. 10:00 pm

This section will elaborate on the data processing process.

We used a consolidated data file of more than 1.8 million records that is available on EIA's SEDS [seds-data-complete website](https://www.eia.gov/state/seds/seds-data-complete.php?sid=US).

**Note**: For this notebook, We used the file that contains records from 1960-2019. The file can be downloaded [here](https://www.eia.gov/state/seds/CDF/Complete_SEDS.csv) 

There is an updated file that contains the data from 1960-2020, available on EIA's SEDS updated [website](https://www.eia.gov/state/seds/seds-data-fuel.php?sid=US) website. The file can be downloaded using this [link](https://www.eia.gov/state/seds/sep_update/Complete_SEDS_update.csv).

In [1]:
# import needed packages
import pandas as pd

### Read and Clean the dataset

In [2]:
# read the dataset dictionary (info, calc)
df = pd.read_excel("info/info.xlsx", sheet_name=['info', 'calc'])
df_info = df["info"]
df_calc = df["calc"]

In [3]:
# read the source dataset
df_data = pd.read_csv("https://www.eia.gov/state/seds/CDF/Complete_SEDS.csv")
df_data

# drop Data_Status
# we don't need it since Data_Status all equals 2019F
# to verify it, split this cell in this line here and run 
# ```
# df_data["Data_Status"].drop_duplicates()
# ```
df_data = df_data.drop(columns="Data_Status")
df_data

# select only needed MSN
df_data = df_data[df_data["MSN"].isin(df_calc["Variable"].to_list())]
df_data

# pivot the dataset
df_data = pd.pivot(df_data, index=["StateCode", "Year"], columns="MSN", values="Data")
df_data

# reset index so that the pivoted dataset becomes a normal table again
df_data.reset_index(inplace=True)
df_data


MSN,StateCode,Year,BDACB,BDLCB,BDTCB,CCEXB,CCIMB,CLACB,CLCCB,CLEIB,...,WDRCB,WDTCB,WSCCB,WSEIB,WSICB,WSTCB,WYCCB,WYEGB,WYICB,WYTCB
0,AK,1960,0.0,0.0,0.0,NaN,NaN,86.0,496.0,914.0,...,1806.0,3681.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AK,1961,0.0,0.0,0.0,NaN,NaN,42.0,496.0,1127.0,...,1823.0,4145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AK,1962,0.0,0.0,0.0,NaN,NaN,43.0,593.0,1373.0,...,1703.0,4246.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AK,1963,0.0,0.0,0.0,NaN,NaN,36.0,393.0,1482.0,...,1651.0,4383.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AK,1964,0.0,0.0,0.0,NaN,NaN,33.0,306.0,2279.0,...,1703.0,4728.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3115,WY,2015,213.0,0.0,213.0,NaN,NaN,0.0,174.0,457661.0,...,4178.0,4904.0,0.0,0.0,2.0,2.0,0.0,35009.0,0.0,35009.0
3116,WY,2016,773.0,0.0,773.0,NaN,NaN,0.0,149.0,425087.0,...,3604.0,4356.0,0.0,0.0,2.0,2.0,0.0,40522.0,0.0,40522.0
3117,WY,2017,505.0,0.0,505.0,NaN,NaN,0.0,285.0,426707.0,...,4189.0,5045.0,0.0,0.0,0.0,0.0,0.0,39806.0,0.0,39806.0
3118,WY,2018,614.0,0.0,614.0,NaN,NaN,0.0,187.0,424234.0,...,4210.0,4929.0,0.0,0.0,0.0,0.0,0.0,36936.0,0.0,36936.0


### Calculate Variables

Our strategy is as follows: for each variable, if all of its sub-variables are already calculated in the dataset, then we calculate that variable. If not, we skip to the next variable.

In [4]:
# transform calc to series of value in dict type
sf_calc = df_calc.groupby(["id"]).apply(
    lambda x: 
    x[["Variable", "Coefficient"]]
    .set_index("Variable")
    .transpose()
    .loc["Coefficient"]
    .to_dict()
)
sf_calc

id
Biodiesel                                                         {'BDTCB': 1}
Biodiesel->Transportation                                         {'BDACB': 1}
BiodieselLoss                                                     {'BDLCB': 1}
BiodieselLoss->Industrial                                         {'BDLCB': 1}
BiodieselSum                              {'Biodiesel': 1, 'BiodieselLoss': 1}
                                                          ...                 
Wood->Residential                                                 {'WDRCB': 1}
WoodProduction                                                    {'WDPRB': 1}
WoodProduction->DensifiedBiomassExport                            {'WDEXB': 1}
WoodProduction->Wood                                              {'WDPRB': 1}
WoodWaste                                              {'Wood': 1, 'Waste': 1}
Length: 96, dtype: object

In [6]:
# calculate the variables

# Our strategy is as follows: for each variable, if all of its sub-variables are already calculated in the dataset, then we calculate that variable. If not, we skip to the next variable.

# number of variables still needed to calculte. By default it equals the number of variables to calculate
to_calculate = len(sf_calc)

while(to_calculate > 0):

    # a dictionary that holds all variables
    d_to_calculate = {}

    for var in sf_calc.index:
        if(var in df_data.columns): 
            sf_calc.drop(var)
            continue

        if(all(x in df_data.columns for x in sf_calc[var].keys())):
            d_to_calculate[var] = sf_calc[var]
            df_data[var] = 0

    to_calculate = len(d_to_calculate)
    
    if(to_calculate == 0): break

    for var in d_to_calculate:
        for subvar in d_to_calculate[var]:
            df_data[var] += df_data[subvar].fillna(0) * d_to_calculate[var][subvar]

df_data

MSN,StateCode,Year,BDACB,BDLCB,BDTCB,CCEXB,CCIMB,CLACB,CLCCB,CLEIB,...,WoodWaste,Biofuel,Biomass,Consumption,FossilFuel,NoncombustibleRenewable,Nonrenewable,Renewable,Total,Clean
0,AK,1960,0.0,0.0,0.0,NaN,NaN,86.0,496.0,914.0,...,3681.0,0.0,3681.0,62445.0,54633.0,3120.0,54633.0,6801.0,61434.0,6801.0
1,AK,1961,0.0,0.0,0.0,NaN,NaN,42.0,496.0,1127.0,...,4145.0,0.0,4145.0,73349.0,64860.0,3168.0,64860.0,7313.0,72173.0,7313.0
2,AK,1962,0.0,0.0,0.0,NaN,NaN,43.0,593.0,1373.0,...,4246.0,0.0,4246.0,80171.0,71359.0,3207.0,71359.0,7453.0,78812.0,7453.0
3,AK,1963,0.0,0.0,0.0,NaN,NaN,36.0,393.0,1482.0,...,4383.0,0.0,4383.0,82436.0,72995.0,3410.0,72995.0,7793.0,80788.0,7793.0
4,AK,1964,0.0,0.0,0.0,NaN,NaN,33.0,306.0,2279.0,...,4728.0,0.0,4728.0,86891.0,76964.0,3374.0,76964.0,8102.0,85066.0,8102.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3115,WY,2015,213.0,0.0,213.0,NaN,NaN,0.0,174.0,457661.0,...,4906.0,3690.0,8596.0,1202644.0,773533.0,43790.0,773533.0,52386.0,825919.0,52386.0
3116,WY,2016,773.0,0.0,773.0,NaN,NaN,0.0,149.0,425087.0,...,4358.0,3948.0,8306.0,1157005.0,744203.0,50205.0,744203.0,58511.0,802714.0,58511.0
3117,WY,2017,505.0,0.0,505.0,NaN,NaN,0.0,285.0,426707.0,...,5045.0,3537.0,8582.0,1186405.0,772548.0,50866.0,772548.0,59448.0,831996.0,59448.0
3118,WY,2018,614.0,0.0,614.0,NaN,NaN,0.0,187.0,424234.0,...,4929.0,3467.0,8396.0,1194080.0,792120.0,46549.0,792120.0,54945.0,847065.0,54945.0


In [7]:
# keep only our needed variables
df_data = df_data[["StateCode", "Year"] + df_info["id"].tolist()]
df_data

MSN,StateCode,Year,WoodProduction,Solar,Wind,Geothermal,Hydropower,Wood,Waste,Biodiesel,...,Wind->ElectricPower,Wind->Commercial,Wood->Residential,Wood->Industrial,Wood->ElectricPower,Wood->Commercial,WoodProduction->Wood,WoodProduction->DensifiedBiomassExport,NetInterstateImport->ElectricPower,ElectricPower->NetInterstateExport
0,AK,1960,3681.0,0.0,0.0,0.0,3120.0,3681.0,0.0,0.0,...,0.0,0.0,1806.0,1840.0,0.0,34.0,3681.0,0.0,0.0,0.0
1,AK,1961,4145.0,0.0,0.0,0.0,3168.0,4145.0,0.0,0.0,...,0.0,0.0,1823.0,2288.0,0.0,35.0,4145.0,0.0,0.0,0.0
2,AK,1962,4246.0,0.0,0.0,0.0,3207.0,4246.0,0.0,0.0,...,0.0,0.0,1703.0,2511.0,0.0,32.0,4246.0,0.0,0.0,0.0
3,AK,1963,4383.0,0.0,0.0,0.0,3410.0,4383.0,0.0,0.0,...,0.0,0.0,1651.0,2700.0,0.0,31.0,4383.0,0.0,0.0,0.0
4,AK,1964,4728.0,0.0,0.0,0.0,3374.0,4728.0,0.0,0.0,...,0.0,0.0,1703.0,2993.0,0.0,32.0,4728.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3115,WY,2015,4904.0,27.0,35009.0,663.0,8091.0,4904.0,2.0,213.0,...,35009.0,0.0,4178.0,113.0,0.0,612.0,4904.0,0.0,-318978.0,318978.0
3116,WY,2016,4356.0,33.0,40522.0,663.0,8987.0,4356.0,2.0,773.0,...,40522.0,0.0,3604.0,113.0,0.0,638.0,4356.0,0.0,-297805.0,297805.0
3117,WY,2017,5045.0,45.0,39806.0,663.0,10352.0,5045.0,0.0,505.0,...,39806.0,0.0,4189.0,88.0,0.0,769.0,5045.0,0.0,-297161.0,297161.0
3118,WY,2018,4929.0,67.0,36936.0,663.0,8883.0,4929.0,0.0,614.0,...,36936.0,0.0,4210.0,87.0,0.0,632.0,4929.0,0.0,-289472.0,289472.0


### Write the dataset to files

In [8]:
# write data to json
df_data.groupby("StateCode").apply(
    lambda x:
    x.set_index("Year", drop=False)
    .to_dict("index")
).to_json("data.json")

In [9]:
# write data to csv
df_data.to_csv("data.csv", index=False)